# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print all record sets and their constituent field `@id` values to see what structured data resources exist. These `@id`s are critical for further referencing and extraction.

In [ ]:
# List all record sets defined in the dataset schema
print('Available Record Sets:')
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For each record set, list the fields with their @ids
print('\nFields for each Record Set:')
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, Fields:")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'N/A')}")
    else:
        print("    No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this example, we'll extract data from the primary protein abundance record set (replace with the actual `@id` from above, e.g., `cr:protein_abundance`).

In [ ]:
# Get all record set @ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# For demo, process all available record sets
for record_set_id in record_sets:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Suppose there is a numeric field called `cr:MW` (Molecular Weight) and a grouping field `cr:accession` (the protein accession ID) in the protein abundance record set. Adjust the `record_set_id`, `numeric_field`, and `group_field` below using the correct `@id` values as found in the overview step. We'll demonstrate filtering and normalization.

In [ ]:
# Example - Adjust the following IDs as per your dataset's actual field names and @ids
record_set_id = record_sets[0] if record_sets else None  # Use the first record set for demonstration
if record_set_id:
    df = dataframes[record_set_id]

    # Attempt to infer a numeric field and a group field
    # Print column list
    print(f"Columns in {record_set_id}: {df.columns.tolist()}")

    # Heuristically try to find fields related to MW, intensity, or abundance
    numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ["mw", "abundance", "intensity", "pep", "coverage", "count"])]
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ["accession", "protein", "category", "group"])]

    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]
    group_field = group_candidates[0] if group_candidates else df.columns[0]

    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if present
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, visualize the distribution of the normalized numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30)
    plt.xlabel(f'Normalized {numeric_field}')
    plt.title(f'Distribution of {numeric_field} (Normalized)')
    plt.show()

    # If there are at least two numeric columns, show a scatter plot
    numeric_cols = filtered_df.select_dtypes(include='number').columns.tolist()
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=filtered_df, x=numeric_cols[0], y=numeric_cols[1])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f'Scatterplot of {numeric_cols[0]} vs {numeric_cols[1]}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration. For this dataset, we've loaded protein abundance data from extracellular vesicles, explored its structure, filtered by molecular weight or abundance, and visualized the resulting distributions. Use the field and record set `@id` notation for all programmatic processing to work robustly with `mlcroissant` packages.